# Train a duration policy in SUMO
**Research / developing phase.** Real SUMO/TraCI simulation, no server or ESP32 commands. The model selects locked duration actions while the axes alternate. The current CV classifier does not produce the queue observations used here.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'traffic_training').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the Train folder or its notebooks subfolder.')
sys.path.insert(0, str(ROOT))
from IPython.display import display, Image


In [ ]:
from traffic_training.sumo_env import SumoConfig
OUTPUT = ROOT / 'outputs' / 'rl'
CONFIG = SumoConfig(horizon_seconds=1800, ns_rate=0.24, ew_rate=0.24, gui=False)
TOTAL_DECISIONS = 50000  # Decisions, NOT SUMO seconds; start smaller to check your setup.
ROLLOUT_DECISIONS = 256
BATCH_SIZE = 64
SEED = 42
DEVICE = 'cpu'


## Verify the simulator and phase contract
The network has one lane per approach and straight movements only. Initial all-red lasts 2 seconds. Each action holds the selected red axis for 10–180 seconds, then applies 3s yellow and 2s all-red. No intermediate action changes that hold. Startup and cutoff behavior are logged explicitly.

In [ ]:
from traffic_training.sumo_env import build_network, TrafficSumoEnv, OBSERVATION_NAMES, command_for_action
print('Network:', build_network(CONFIG.network_dir))
print('Observation order:', OBSERVATION_NAMES)
env = TrafficSumoEnv(CONFIG)
try:
    observation, info = env.reset(seed=SEED)
    action = 0
    print('Offline preview:', command_for_action(action, 'north_south', CONFIG.durations))
    observation, reward, terminated, truncated, info = env.step(action)
    display(info)
finally:
    env.close()


## Train PPO
Scenarios and episode seeds randomize within training only. Reward penalizes accumulated halted and pending vehicle-seconds. Gamma and lambda are 1 to avoid inconsistent discounts across unequal hold lengths. This cell can take substantial time; stop/reduce decisions for a setup check, then use fresh output folders for actual experiments.

In [ ]:
from traffic_training.rl import train_policy
metadata = train_policy(OUTPUT, config=CONFIG, total_timesteps=TOTAL_DECISIONS,
    seed=SEED, n_steps=ROLLOUT_DECISIONS, batch_size=BATCH_SIZE, device=DEVICE)
display(metadata)


In [ ]:
import pandas as pd
episodes = pd.read_csv(OUTPUT / 'episodes.csv.monitor.csv', comment='#')
display(episodes.tail())
if len(episodes):
    episodes['r'].plot(title='Training episode return (not held-out performance)', xlabel='Episode', ylabel='Return')
print('Policy:', OUTPUT / 'policy.zip')
print('Keep policy_metadata.json beside the policy.')


Use notebook 04 to compare with fixed timers on held-out seeds. A short successful run verifies the pipeline only. It does not demonstrate a useful trained policy. Live control, the CV observation bridge and real-junction deployment remain under development.